<a href="https://colab.research.google.com/github/vdubya/criteria-assistant/blob/main/NEPATEC_Download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ✅ Download gated HF dataset and split into standalone ZIPs (~4GB each)
#    - Skips .cache and .gitattributes
#    - Uses HF_TOKEN from Colab Secrets (Tools → Secrets)
#    - Each ZIP is independent; extracting all rebuilds the folder tree

!pip -q install -U "huggingface_hub>=0.23.0"

import os, pathlib, zipfile
from google.colab import userdata
from huggingface_hub import snapshot_download

# -------------------
# Config
# -------------------
REPO_ID    = "PNNL/NEPATEC2.0"              # change if needed
LOCAL_DIR  = "/content/NEPATEC2.0"
OUT_DIR    = "/content/zips"
ZIP_PREFIX = "NEPATEC2.0_part"
MAX_ZIP_BYTES = 4 * 1024**3 - 16 * 1024**2   # ~4 GB minus margin

# -------------------
# Token from keystore
# -------------------
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("❌ No HF_TOKEN in Colab Secrets. Add it via Tools → Secrets (must allow gated repos).")

# -------------------
# Download dataset
# -------------------
path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DIR,
    token=HF_TOKEN,
)
print(f"✅ Downloaded to: {path}")

# -------------------
# Collect files, skipping unwanted ones
# -------------------
root = pathlib.Path(LOCAL_DIR)
files = [
    p for p in root.rglob("*")
    if p.is_file() and
       ".cache" not in p.parts and
       ".gitattributes" not in p.name
]
files.sort(key=lambda p: p.stat().st_size)
print(f"📦 Files to include: {len(files):,}")

# -------------------
# ZIP creation
# -------------------
os.makedirs(OUT_DIR, exist_ok=True)
part_idx, current_bytes = 1, 0
zip_path = pathlib.Path(OUT_DIR) / f"{ZIP_PREFIX}{part_idx:03d}.zip"
zf = zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, allowZip64=True)
created = [zip_path]

for fp in files:
    rel = fp.relative_to(root)
    fsize = fp.stat().st_size
    if current_bytes + fsize > MAX_ZIP_BYTES and current_bytes > 0:
        zf.close()
        part_idx += 1
        zip_path = pathlib.Path(OUT_DIR) / f"{ZIP_PREFIX}{part_idx:03d}.zip"
        zf = zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, allowZip64=True)
        created.append(zip_path)
        current_bytes = 0
    zf.write(fp, arcname=str(rel))
    current_bytes += fsize

zf.close()

# -------------------
# Report
# -------------------
print("\n✅ Created standalone ZIPs (≤ ~4 GB each, skipping .cache and .gitattributes):")
for z in created:
    if z.exists():
        print(f"  {z.name:30s}  {z.stat().st_size/1_048_576:10.1f} MB")

print("\n📂 Output folder:", OUT_DIR)
print("➡ Unzip all ZIPs into the same folder to reconstruct the dataset.")
print("   Linux/macOS:  mkdir out && for z in /content/zips/*.zip; do unzip -o \"$z\" -d out; done")
print("   Windows (PowerShell):  foreach ($z in Get-ChildItem zips\\*.zip) { Expand-Archive -Path $z -DestinationPath out -Force }")


Fetching 507 files:   0%|          | 0/507 [00:00<?, ?it/s]

✅ Downloaded to: /content/NEPATEC2.0
📦 Files to include: 506

✅ Created standalone ZIPs (≤ ~4 GB each, skipping .cache and .gitattributes):
  NEPATEC2.0_part001.zip              1003.6 MB
  NEPATEC2.0_part002.zip               965.6 MB
  NEPATEC2.0_part003.zip               903.8 MB
  NEPATEC2.0_part004.zip               797.6 MB
  NEPATEC2.0_part005.zip               327.5 MB

📂 Output folder: /content/zips
➡ Unzip all ZIPs into the same folder to reconstruct the dataset.
   Linux/macOS:  mkdir out && for z in /content/zips/*.zip; do unzip -o "$z" -d out; done
   Windows (PowerShell):  foreach ($z in Get-ChildItem zips\*.zip) { Expand-Archive -Path $z -DestinationPath out -Force }


In [ ]:
# ✅ Copy all created ZIPs from /content/zips → Google Drive for safe download
from google.colab import drive
import pathlib, shutil

# 1️⃣ Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2️⃣ Define source and destination
SRC_DIR  = pathlib.Path("/content/zips")                     # where your ZIPs currently live
DEST_DIR = pathlib.Path("/content/drive/MyDrive/NEPATEC2.0_zips")  # new folder in Drive

# 3️⃣ Create destination folder if it doesn't exist
DEST_DIR.mkdir(parents=True, exist_ok=True)

# 4️⃣ Copy all .zip files
copied = []
for p in SRC_DIR.glob("*.zip"):
    dest = DEST_DIR / p.name
    shutil.copy2(p, dest)
    copied.append(dest)

# 5️⃣ Report
print(f"✅ Copied {len(copied)} ZIPs to Google Drive:")
for f in copied:
    print("  •", f)
print("\n📂 You can now download them from:")
print("   Google Drive → My Drive → NEPATEC2.0_zips")


Mounted at /content/drive
✅ Copied 5 ZIPs to Google Drive:
  • /content/drive/MyDrive/NEPATEC2.0_zips/NEPATEC2.0_part005.zip
  • /content/drive/MyDrive/NEPATEC2.0_zips/NEPATEC2.0_part004.zip
  • /content/drive/MyDrive/NEPATEC2.0_zips/NEPATEC2.0_part003.zip
  • /content/drive/MyDrive/NEPATEC2.0_zips/NEPATEC2.0_part001.zip
  • /content/drive/MyDrive/NEPATEC2.0_zips/NEPATEC2.0_part002.zip

📂 You can now download them from:
   Google Drive → My Drive → NEPATEC2.0_zips
